# 逻辑回归机器学习选股策略

## 完整流程

| 步骤 | 内容 | 关键技术 |
|------|------|----------|
| 1 | 读取本地股票数据 | pandas CSV 加载 |
| 2 | 按板块/逻辑筛选标的 | 主板过滤、ST排除、OOS保留 |
| 3 | 构建过去X年特征数据集 | TechnicalFactorCalculator |
| 4 | 划分训练/验证/测试集 | 严格时序划分，无数据穿越 |
| 5 | 训练逻辑回归模型 | sklearn Pipeline + L2正则 |
| 6 | 新标的回测效果 | VectorBT 信号回测 |

> **核心原则**: 严格防止数据穿越（Look-Ahead Bias）——目标标签、特征缩放参数、超参数均只能从训练集计算。

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# 项目路径
import pathlib
PROJECT_ROOT = pathlib.Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from datetime import datetime, timedelta

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, average_precision_score,
    precision_recall_curve, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

try:
    import vectorbt as vbt
    HAS_VBT = True
except ImportError:
    HAS_VBT = False
    print("⚠️  vectorbt 未安装，回测步骤将使用简化实现")

from utils.factor_calculator import TechnicalFactorCalculator
from utils.constants import (
    DEFAULT_COMMISSION_RATE, DEFAULT_SLIPPAGE,
    DEFAULT_TAX_RATE, TRADING_DAYS_PER_YEAR
)

# ============================================================
# 全局参数（所有可调参数集中在此处）
# ============================================================
CSV_PATH         = './data/a_stock_history_price.csv'
HISTORY_YEARS    = 5        # 使用过去 N 年数据
PRED_HORIZON     = 5        # 预测未来 N 天的涨跌
RETURN_THRESHOLD = 0.02     # 正样本阈值：涨幅 > 2%
TRAIN_RATIO      = 0.60
VAL_RATIO        = 0.20
TEST_RATIO       = 0.20
OOS_HOLDOUT      = 0.20     # 20% 股票完全保留用于 OOS 回测
PROB_THRESHOLD   = 0.55     # 信号触发概率阈值
RANDOM_SEED      = 42

plt.rcParams['font.sans-serif'] = ['SimHei', 'PingFang SC', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

print("✅ 环境配置完成")
print(f"   预测目标: 未来 {PRED_HORIZON} 日涨幅 > {RETURN_THRESHOLD*100:.0f}%")
print(f"   训练/验证/测试: {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")
print(f"   OOS 保留比例: {OOS_HOLDOUT:.0%}")

## 步骤 1：读取本地股票数据

从本地 CSV 文件加载 A 股历史数据。

**预期列名**（来自 `load_a_stock_data.py`）:
- `stock_code` — 股票代码（可能为整数，需零填充至6位）
- `name` — 股票名称
- `date` — 交易日期
- `open`, `high`, `low`, `close` — OHLC 价格
- `volume` — 成交量
- `amount` — 成交额
- `price_change_rate` — 日涨跌幅（%）

In [ ]:
class LocalCSVLoader:
    """
    从本地 CSV 文件加载 A 股历史数据。
    
    自动处理:
    - stock_code 零填充（整数 1 → '000001'）
    - 列名标准化
    - 日期类型转换
    - 数据范围过滤
    """
    
    EXPECTED_COLS = ['date', 'open', 'high', 'low', 'close', 'volume']
    
    def __init__(self, path: str):
        self.path = path
    
    def load(self, years_back: int = None) -> pd.DataFrame:
        """加载并标准化数据"""
        if not os.path.exists(self.path):
            raise FileNotFoundError(
                f"找不到数据文件: {self.path}\n"
                f"请确认文件路径，或将 CSV_PATH 修改为正确路径"
            )
        
        print(f"📂 加载数据: {self.path}")
        df = pd.read_csv(self.path, low_memory=False)
        print(f"   原始: {df.shape[0]:,} 行 × {df.shape[1]} 列")
        
        # 标准化列名
        df = self._normalize_columns(df)
        
        # 日期处理
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['date'])
        
        # 数值列处理
        num_cols = ['open', 'high', 'low', 'close', 'volume', 'amount', 'price_change_rate']
        for col in num_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 按年份过滤
        if years_back is not None:
            cutoff = pd.Timestamp.now() - pd.DateOffset(years=years_back)
            df = df[df['date'] >= cutoff]
        
        # 排序
        df = df.sort_values(['code', 'date']).reset_index(drop=True)
        
        mem_mb = df.memory_usage(deep=True).sum() / 1024**2
        print(f"   加载后: {df.shape[0]:,} 行 | {df['code'].nunique():,} 只股票")
        print(f"   日期范围: {df['date'].min().date()} ~ {df['date'].max().date()}")
        print(f"   内存占用: {mem_mb:.1f} MB")
        return df
    
    def _normalize_columns(self, df: pd.DataFrame) -> pd.DataFrame:
        """统一列名"""
        col_map = {
            'stock_code': 'code',
            'ts_code': 'code',
            'trade_date': 'date',
            'price_change_rate': 'pct_change_raw',  # 保留原始列
            'vol': 'volume',
            'turnover': 'amount',
        }
        df = df.rename(columns=col_map)
        
        # 零填充股票代码
        if 'code' in df.columns:
            df['code'] = df['code'].astype(str).str.zfill(6)
        
        # 保留 price_change_rate 的兼容名
        if 'pct_change_raw' in df.columns and 'price_change_rate' not in df.columns:
            df['price_change_rate'] = df['pct_change_raw']
        
        # 添加 pct_change（计算得出）
        if 'pct_change' not in df.columns and 'close' in df.columns:
            df['pct_change'] = df.groupby('code')['close'].pct_change()
        
        return df
    
    def _get_exchange(self, code: str) -> str:
        """判断交易所"""
        if code.startswith('6') or code.startswith('5'):
            return 'SH'
        elif code.startswith('30'):
            return 'CY'  # 创业板
        elif code.startswith('68'):
            return 'KC'  # 科创板
        elif code.startswith('8') or code.startswith('4'):
            return 'BJ'  # 北交所
        else:
            return 'SZ'  # 深交所主板

In [ ]:
# 加载数据
loader = LocalCSVLoader(CSV_PATH)
df_raw = loader.load(years_back=HISTORY_YEARS)

# 添加交易所信息
df_raw['exchange'] = df_raw['code'].apply(loader._get_exchange)

# 展示样例
print("\n📋 数据样例（前5行）:")
display_cols = [c for c in ['code', 'date', 'open', 'high', 'low', 'close', 'volume', 'exchange'] 
                if c in df_raw.columns]
df_raw[display_cols].head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 每年记录数
yearly = df_raw.groupby(df_raw['date'].dt.year).size()
axes[0].bar(yearly.index, yearly.values, color='steelblue', alpha=0.8)
axes[0].set_title('每年数据量（行数）', fontsize=13)
axes[0].set_xlabel('年份')
axes[0].set_ylabel('记录数')
for i, (yr, cnt) in enumerate(yearly.items()):
    axes[0].text(yr, cnt + yearly.max()*0.01, f'{cnt//10000:.0f}w', 
                 ha='center', fontsize=8)

# 各交易所占比
exch_cnt = df_raw.groupby('exchange')['code'].nunique()
axes[1].pie(exch_cnt.values, labels=exch_cnt.index, autopct='%1.1f%%', 
            colors=['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0'])
axes[1].set_title('股票数量（按交易所）', fontsize=13)

plt.suptitle('步骤1: 数据概览', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 步骤 2：标的筛选

**筛选策略**（A 股常用逻辑）:

1. **板块过滤**: 仅保留主板（SH + SZ），排除创业板、科创板、北交所
2. **ST 排除**: 剔除 ST / \*ST 股票（退市风险，涨跌停限制不同）
3. **数据充足性**: 近 N 年内有效交易日 ≥ 504（约2年），保证特征计算可靠
4. **OOS 保留**: 随机保留 20% 股票作为纯样本外（OOS）回测标的，完全不参与训练

In [ ]:
def filter_universe(df: pd.DataFrame,
                    boards=('SH', 'SZ'),
                    exclude_st=True,
                    min_trading_days=504,
                    oos_holdout=0.20,
                    random_seed=42) -> tuple:
    """
    筛选股票池，并划分 ML训练池 和 OOS回测池。
    
    Returns:
        ml_codes   (list): 用于特征构建和模型训练的股票代码
        oos_codes  (list): 完全保留给OOS回测的股票代码（不参与训练）
        df_ml      (DataFrame): ml_codes 对应的历史数据
        df_oos     (DataFrame): oos_codes 对应的历史数据
        report     (dict): 筛选统计报告
    """
    report = {'raw_stocks': df['code'].nunique()}
    
    # Step 1: 板块过滤
    df1 = df[df['exchange'].isin(boards)].copy()
    report['after_board_filter'] = df1['code'].nunique()
    
    # Step 2: ST排除
    if exclude_st and 'name' in df1.columns:
        st_codes = df1[df1['name'].str.contains('ST', na=False)]['code'].unique()
        df1 = df1[~df1['code'].isin(st_codes)]
        report['st_excluded'] = len(st_codes)
    else:
        report['st_excluded'] = 0
    report['after_st_filter'] = df1['code'].nunique()
    
    # Step 3: 数据充足性
    day_counts = df1.groupby('code')['date'].count()
    valid_codes = day_counts[day_counts >= min_trading_days].index.tolist()
    df1 = df1[df1['code'].isin(valid_codes)]
    report['after_data_filter'] = len(valid_codes)
    
    # Step 4: OOS/ML 划分
    np.random.seed(random_seed)
    shuffled = np.random.permutation(valid_codes)
    n_oos = int(len(shuffled) * oos_holdout)
    oos_codes = list(shuffled[:n_oos])
    ml_codes  = list(shuffled[n_oos:])
    report['oos_count'] = len(oos_codes)
    report['ml_count']  = len(ml_codes)
    
    df_ml  = df1[df1['code'].isin(ml_codes)].copy()
    df_oos = df1[df1['code'].isin(oos_codes)].copy()
    
    return ml_codes, oos_codes, df_ml, df_oos, report

In [ ]:
ml_codes, oos_codes, df_ml, df_oos, filter_report = filter_universe(
    df_raw,
    oos_holdout=OOS_HOLDOUT,
    random_seed=RANDOM_SEED
)

print("=" * 50)
print("📊 标的筛选报告")
print("=" * 50)
print(f"  原始股票数:        {filter_report['raw_stocks']:>6,}")
print(f"  板块过滤后:        {filter_report['after_board_filter']:>6,}")
print(f"  排除ST后 (-{filter_report['st_excluded']}):   {filter_report['after_st_filter']:>6,}")
print(f"  数据充足后:        {filter_report['after_data_filter']:>6,}")
print(f"  ─────────────────────────────")
print(f"  ML训练池:          {filter_report['ml_count']:>6,}")
print(f"  OOS回测池:         {filter_report['oos_count']:>6,}  (OOS={OOS_HOLDOUT:.0%})")
print("=" * 50)
print(f"\n⚠️  OOS股票完全不参与任何训练步骤，保证样本外公正性")

In [ ]:
# 每只股票的有效数据天数
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# ML池与OOS池 数据天数分布
ml_days   = df_ml.groupby('code')['date'].count().sort_values(ascending=False)
oos_days  = df_oos.groupby('code')['date'].count().sort_values(ascending=False)

axes[0].hist(ml_days.values, bins=30, alpha=0.7, label=f'ML训练池 (n={len(ml_days)})', color='steelblue')
axes[0].hist(oos_days.values, bins=30, alpha=0.7, label=f'OOS回测池 (n={len(oos_days)})', color='coral')
axes[0].set_title('各股票有效交易日分布')
axes[0].set_xlabel('交易日数')
axes[0].set_ylabel('股票数量')
axes[0].legend()
axes[0].axvline(504, color='red', linestyle='--', label='最低门槛(504天)')

# Top20 ML池股票（数据最多）
top20 = ml_days.head(20)
axes[1].barh(range(20), top20.values[::-1], color='steelblue', alpha=0.8)
axes[1].set_yticks(range(20))
axes[1].set_yticklabels(top20.index[::-1], fontsize=8)
axes[1].set_title('ML训练池 - 数据最多的20只股票')
axes[1].set_xlabel('交易日数')

plt.suptitle('步骤2: 标的筛选结果', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 步骤 3：构建特征数据集（过去 X 年）

对每只 ML 训练池股票计算技术指标特征，堆叠成面板 DataFrame。

**特征类别**（约 40 个）:

| 类别 | 特征 | 工具方法 |
|------|------|----------|
| 均线系统 | price_to_ma_*, ma_*_slope, ma_bull_arrange | `moving_average` |
| RSI | rsi_6, rsi_12, rsi_24 | `rsi` |
| MACD | macd_dif, macd_histogram, macd_golden_cross | `macd` |
| 布林带 | bb_position, bb_width, bb_break_* | `bollinger_bands` |
| 波动率 | volatility_20/60, amplitude_*, price_position_20 | `volatility_factors` |
| 成交量 | volume_ratio, volume_price_corr, price_vol_up | `volume_factors` |
| 动量 | momentum_5/10/20/60, momentum_st_lt | `momentum_factors` |
| KDJ | kdj_k/d/j, kdj_golden_cross, kdj_j_oversold | `kdj` |

**目标标签**: `label = 1` 当未来 {PRED_HORIZON} 日累计涨幅 > {RETURN_THRESHOLD*100:.0f}%

In [ ]:
def build_features_single(stock_df: pd.DataFrame) -> pd.DataFrame:
    """
    对单只股票计算全部技术因子，并附加目标标签。
    
    严格无穿越设计:
    - 所有 rolling/ewm 只用历史数据（pandas 默认行为）
    - label 用 shift(-N)，但标签行在 split 前已存在，不影响时序划分
    - 尾部 PRED_HORIZON 行（无法确认标签）被删除
    
    Parameters:
        stock_df: 单只股票的 OHLCV DataFrame，含 'date', 'open', 'high', 'low', 'close', 'volume', 'amount'
    
    Returns:
        含特征列和 label 列的 DataFrame
    """
    df = stock_df.copy().sort_values('date').reset_index(drop=True)
    
    # 检查必要列
    required = ['open', 'high', 'low', 'close', 'volume']
    missing = [c for c in required if c not in df.columns]
    if missing:
        return None  # 数据不完整，跳过
    
    if len(df) < 100:
        return None  # 数据太少
    
    calc = TechnicalFactorCalculator()
    
    # 均线系统
    df = calc.moving_average(df, windows=[5, 10, 20, 60])
    # RSI
    df = calc.rsi(df, windows=[6, 12, 24])
    # MACD
    df = calc.macd(df)
    # 布林带
    df = calc.bollinger_bands(df)
    # 波动率因子
    df = calc.volatility_factors(df, windows=[20, 60])
    # 成交量因子
    df = calc.volume_factors(df)
    # 动量因子
    df = calc.momentum_factors(df, windows=[5, 10, 20, 60])
    # KDJ
    df = calc.kdj(df)
    
    # ============ 目标标签（无穿越）============
    # future_return = (未来第N天收盘 / 当前收盘) - 1
    # shift(-N) 意味着当前行使用未来数据，但这在 split 前已确定，
    # 只要 split 严格按时间轴划分，训练集就不会看到测试集的标签
    future_return = df['close'].shift(-PRED_HORIZON) / df['close'] - 1
    df['label'] = (future_return > RETURN_THRESHOLD).astype(int)
    df['future_return'] = future_return
    
    # 删除尾部无标签行
    df = df.dropna(subset=['label'])
    
    # 删除头部因子窗口期 NaN 行（保留更干净的数据）
    df = df.dropna(subset=['ma_60', 'rsi_24', 'volatility_60'])
    
    return df

In [ ]:
print("🔄 构建特征面板（可能需要几分钟...）")
all_dfs = []
failed = 0

for i, code in enumerate(ml_codes):
    stock_df = df_ml[df_ml['code'] == code].copy()
    feat_df = build_features_single(stock_df)
    if feat_df is not None and len(feat_df) > 100:
        all_dfs.append(feat_df)
    else:
        failed += 1
    
    if (i + 1) % 100 == 0:
        print(f"  已处理 {i+1}/{len(ml_codes)} 只股票...")

panel_df = pd.concat(all_dfs, ignore_index=True)
panel_df = panel_df.sort_values(['code', 'date']).reset_index(drop=True)

print(f"\n✅ 面板构建完成")
print(f"   成功: {len(all_dfs)} 只股票 | 跳过: {failed} 只")
print(f"   面板大小: {panel_df.shape[0]:,} 行 × {panel_df.shape[1]} 列")
print(f"   日期范围: {panel_df['date'].min().date()} ~ {panel_df['date'].max().date()}")
print(f"\n📊 标签分布:")
label_counts = panel_df['label'].value_counts()
print(f"   正样本 (涨幅>{RETURN_THRESHOLD*100:.0f}%): {label_counts.get(1,0):,} ({label_counts.get(1,0)/len(panel_df)*100:.1f}%)")
print(f"   负样本:                      {label_counts.get(0,0):,} ({label_counts.get(0,0)/len(panel_df)*100:.1f}%)")

In [ ]:
# 定义特征列（在此统一确定，后续步骤均使用此变量）
META_COLS = ['date', 'code', 'name', 'open', 'high', 'low', 'close',
             'volume', 'amount', 'price_change_rate', 'pct_change', 
             'pct_change_raw', 'exchange', 'label', 'future_return',
             'ma_5', 'ma_10', 'ma_20', 'ma_60',  # 原始均线（非因子）
             'bb_upper', 'bb_lower', 'bb_middle',  # 原始布林带
             'macd_dif', 'macd_dea',               # MACD 中间值
             'kdj_k', 'kdj_d', 'kdj_j',            # KDJ 原始值
             'high_20', 'low_20',                  # 高低点
             'amount_ma_5', 'amount_ma_20']         # 成交额均线

FEATURE_COLS = [c for c in panel_df.columns 
                if c not in META_COLS and panel_df[c].dtype in [np.float64, np.int64, np.float32, np.int32]]

print(f"📌 特征数量: {len(FEATURE_COLS)} 个")
print("特征列表:", FEATURE_COLS)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 标签分布
label_counts = panel_df['label'].value_counts()
axes[0].bar(['负样本(0)\n涨幅≤2%', f'正样本(1)\n涨幅>{RETURN_THRESHOLD*100:.0f}%'],
            [label_counts.get(0, 0), label_counts.get(1, 0)],
            color=['#E57373', '#81C784'])
axes[0].set_title('目标标签分布（类别平衡）')
axes[0].set_ylabel('样本数量')
for i, v in enumerate([label_counts.get(0, 0), label_counts.get(1, 0)]):
    axes[0].text(i, v + 1000, f'{v:,}\n({v/len(panel_df)*100:.1f}%)', ha='center')

# 特征与标签的 Spearman 相关系数（信息系数 IC）
sample_df = panel_df.sample(min(50000, len(panel_df)), random_state=42)
ic_vals = {}
for col in FEATURE_COLS[:25]:  # 前25个特征
    ic = sample_df[col].corr(sample_df['label'], method='spearman')
    ic_vals[col] = ic

ic_series = pd.Series(ic_vals).sort_values(key=abs, ascending=False)
colors = ['#EF5350' if v > 0 else '#42A5F5' for v in ic_series.values]
axes[1].barh(range(len(ic_series)), ic_series.values[::-1], color=colors[::-1])
axes[1].set_yticks(range(len(ic_series)))
axes[1].set_yticklabels(ic_series.index[::-1], fontsize=7)
axes[1].set_title('特征信息系数（IC = Spearman相关系数）')
axes[1].set_xlabel('IC值')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('步骤3: 特征数据集概览', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 步骤 4：划分训练集、验证集和测试集

**设计原则（严格防穿越）**:
- 按**全局日期轴**划分，而非随机划分
- 所有股票的训练数据日期 < 验证数据日期 < 测试数据日期
- 不使用 sklearn 的 `KFold`（会打乱时序）

**比例**: 训练 60% | 验证 20% | 测试 20%

In [ ]:
class PanelTimeSeriesSplitter:
    """
    面板数据时间序列划分器。
    
    按全局日期轴（所有股票共享）进行划分，确保:
    1. 训练集最晚日期 < 验证集最早日期 < 测试集最早日期
    2. 相同股票在不同分段中均有数据
    """
    
    def split(self, df, feature_cols,
              train_ratio=0.60, val_ratio=0.20):
        """
        按日期轴划分面板数据。
        
        Returns:
            (X_train, y_train, df_train),
            (X_val,   y_val,   df_val),
            (X_test,  y_test,  df_test)
        """
        # 获取全局唯一日期序列
        all_dates = sorted(df['date'].unique())
        n = len(all_dates)
        
        train_end_idx = int(n * train_ratio)
        val_end_idx   = int(n * (train_ratio + val_ratio))
        
        self.train_end_date = all_dates[train_end_idx - 1]
        self.val_end_date   = all_dates[val_end_idx - 1]
        self.test_start_date = all_dates[val_end_idx]
        
        df_train = df[df['date'] <= self.train_end_date].copy()
        df_val   = df[(df['date'] > self.train_end_date) & 
                      (df['date'] <= self.val_end_date)].copy()
        df_test  = df[df['date'] > self.val_end_date].copy()
        
        def extract(split_df):
            X = split_df[feature_cols].fillna(0)
            y = split_df['label']
            return X, y, split_df
        
        return extract(df_train), extract(df_val), extract(df_test)


splitter = PanelTimeSeriesSplitter()
(X_train, y_train, df_train), \
(X_val,   y_val,   df_val),   \
(X_test,  y_test,  df_test) = splitter.split(panel_df, FEATURE_COLS)

print("=" * 60)
print("📅 时间序列划分结果")
print("=" * 60)
for name, split_df, y in [('训练集', df_train, y_train),
                            ('验证集', df_val,   y_val),
                            ('测试集', df_test,  y_test)]:
    pos_rate = y.mean()
    print(f"  {name}:")
    print(f"    日期范围: {split_df['date'].min().date()} ~ {split_df['date'].max().date()}")
    print(f"    样本数:   {len(split_df):>8,}")
    print(f"    正样本率: {pos_rate:.1%}")
    print()

In [ ]:
# 可视化时序划分
fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# 时间轴色块图
splits = [
    (df_train['date'].min(), df_train['date'].max(), '#4CAF50', f'训练集\n{len(df_train):,}行'),
    (df_val['date'].min(),   df_val['date'].max(),   '#FF9800', f'验证集\n{len(df_val):,}行'),
    (df_test['date'].min(),  df_test['date'].max(),  '#F44336', f'测试集\n{len(df_test):,}行'),
]
for start, end, color, label in splits:
    axes[0].barh(0, (end - start).days, left=start, height=0.4, 
                 color=color, alpha=0.8, label=label)
    mid = start + (end - start) / 2
    axes[0].text(mid, 0, label, ha='center', va='center', fontsize=9, fontweight='bold')

axes[0].set_title('时间序列划分（严格无穿越）', fontsize=13)
axes[0].set_yticks([])
axes[0].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
axes[0].set_xlim(panel_df['date'].min(), panel_df['date'].max())

# 各分段正样本率
split_stats = pd.DataFrame({
    '分段': ['训练集', '验证集', '测试集'],
    '正样本率': [y_train.mean(), y_val.mean(), y_test.mean()],
    '样本数': [len(y_train), len(y_val), len(y_test)]
})
colors_bar = ['#4CAF50', '#FF9800', '#F44336']
axes[1].bar(split_stats['分段'], split_stats['正样本率'], color=colors_bar, alpha=0.8)
axes[1].set_title('各分段正样本率（检查类别分布稳定性）')
axes[1].set_ylabel('正样本比例')
axes[1].axhline(y_train.mean(), color='green', linestyle='--', alpha=0.5, label=f'训练集均值 {y_train.mean():.2%}')
for i, (rate, n) in enumerate(zip(split_stats['正样本率'], split_stats['样本数'])):
    axes[1].text(i, rate + 0.002, f'{rate:.1%}\n({n:,})', ha='center', fontsize=9)

plt.suptitle('步骤4: 训练/验证/测试集划分', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 步骤 5：训练逻辑回归模型

**模型配置**:
- `penalty='l2'`（岭回归正则）：处理相关特征，防过拟合
- `class_weight='balanced'`：应对类别不平衡
- `solver='lbfgs'`：适合 L2 + 中大型数据集
- `StandardScaler`：装入 sklearn Pipeline，仅 fit 训练集（防穿越）

**超参数搜索**：在验证集上搜索最优正则强度 C（不用 KFold 防时序穿越）

**评估指标**：ROC-AUC（对阈值无关，适合金融概率预测）

In [ ]:
def winsorize_fit_transform(X_train, X_val, X_test, n_mad=3):
    """
    MAD 缩尾：仅从训练集计算边界，然后应用到所有分段。
    防止验证集/测试集的极端值影响特征分布。
    """
    X_tr = X_train.copy()
    X_v  = X_val.copy()
    X_te = X_test.copy()
    
    for col in X_tr.columns:
        median = X_tr[col].median()
        mad    = np.median(np.abs(X_tr[col] - median))
        scale  = 1.4826 * mad  # MAD → std 估计量
        upper  = median + n_mad * scale
        lower  = median - n_mad * scale
        for X in [X_tr, X_v, X_te]:
            X[col] = X[col].clip(lower, upper)
    
    return X_tr, X_v, X_te


# 特征预处理（防穿越：只用训练集统计量）
print("🔄 特征预处理（MAD缩尾）...")
X_train_w, X_val_w, X_test_w = winsorize_fit_transform(X_train, X_val, X_test)
print("✅ 完成")

In [ ]:
# 超参数搜索：在验证集上找最优 C
print("🔍 超参数搜索（C值）...")
param_candidates = [0.001, 0.01, 0.1, 1.0, 10.0]
val_results = []

for C in param_candidates:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='l2',
            C=C,
            class_weight='balanced',
            solver='lbfgs',
            max_iter=500,
            random_state=RANDOM_SEED
        ))
    ])
    pipe.fit(X_train_w, y_train)
    val_prob = pipe.predict_proba(X_val_w)[:, 1]
    val_auc  = roc_auc_score(y_val, val_prob)
    val_results.append({'C': C, 'val_auc': val_auc, 'model': pipe})
    print(f"  C={C:6.3f}  验证集AUC={val_auc:.4f}")

# 选择最优模型
best_result = max(val_results, key=lambda x: x['val_auc'])
best_model  = best_result['model']
best_C      = best_result['C']

print(f"\n🏆 最优 C = {best_C}，验证集 AUC = {best_result['val_auc']:.4f}")

In [ ]:
# 验证集详细评估
val_prob  = best_model.predict_proba(X_val_w)[:, 1]
val_pred  = best_model.predict(X_val_w)
val_auc   = roc_auc_score(y_val, val_prob)
val_ap    = average_precision_score(y_val, val_prob)

fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# 1. AUC 搜索曲线
ax1 = fig.add_subplot(gs[0, 0])
C_vals = [r['C'] for r in val_results]
auc_vals = [r['val_auc'] for r in val_results]
ax1.semilogx(C_vals, auc_vals, 'bo-', linewidth=2)
ax1.axvline(best_C, color='red', linestyle='--', label=f'最优 C={best_C}')
ax1.set_title('超参数搜索：C vs 验证集AUC')
ax1.set_xlabel('C（正则强度倒数）')
ax1.set_ylabel('ROC-AUC')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 混淆矩阵
ax2 = fig.add_subplot(gs[0, 1])
cm = confusion_matrix(y_val, val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['负样本(0)', '正样本(1)'])
disp.plot(ax=ax2, colorbar=False, cmap='Blues')
ax2.set_title(f'验证集混淆矩阵\nAUC={val_auc:.4f}')

# 3. ROC 曲线
ax3 = fig.add_subplot(gs[0, 2])
fpr, tpr, _ = roc_curve(y_val, val_prob)
ax3.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC={val_auc:.3f})')
ax3.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='随机猜测')
ax3.fill_between(fpr, tpr, alpha=0.1)
ax3.set_title('ROC 曲线（验证集）')
ax3.set_xlabel('假正率 (FPR)')
ax3.set_ylabel('真正率 (TPR)')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. PR 曲线
ax4 = fig.add_subplot(gs[1, 0])
prec, rec, _ = precision_recall_curve(y_val, val_prob)
ax4.plot(rec, prec, 'g-', linewidth=2, label=f'PR曲线 (AP={val_ap:.3f})')
ax4.axhline(y_val.mean(), color='red', linestyle='--', alpha=0.7, label=f'基线={y_val.mean():.2f}')
ax4.set_title('Precision-Recall 曲线（验证集）')
ax4.set_xlabel('召回率 (Recall)')
ax4.set_ylabel('精确率 (Precision)')
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. 特征系数
ax5 = fig.add_subplot(gs[1, 1:])
coef = pd.Series(
    best_model.named_steps['clf'].coef_[0],
    index=FEATURE_COLS
).sort_values(key=abs, ascending=False).head(20)
colors_coef = ['#EF5350' if v > 0 else '#42A5F5' for v in coef.values[::-1]]
ax5.barh(range(len(coef)), coef.values[::-1], color=colors_coef)
ax5.set_yticks(range(len(coef)))
ax5.set_yticklabels(coef.index[::-1], fontsize=8)
ax5.set_title('Top 20 特征系数（正=利好 负=利空）')
ax5.axvline(0, color='black', linewidth=0.8)
ax5.set_xlabel('系数值（L2正则后）')

plt.suptitle('步骤5: 逻辑回归模型评估（验证集）', fontsize=14, fontweight='bold')
plt.show()
print(f"\n分类报告（验证集）:\n{classification_report(y_val, val_pred, target_names=['负样本', '正样本'])}")

In [ ]:
# 测试集评估（真正的 Holdout）
test_prob = best_model.predict_proba(X_test_w)[:, 1]
test_pred = best_model.predict(X_test_w)
test_auc  = roc_auc_score(y_test, test_prob)
test_ap   = average_precision_score(y_test, test_prob)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 混淆矩阵
cm_test = confusion_matrix(y_test, test_pred)
ConfusionMatrixDisplay(cm_test, display_labels=['负样本', '正样本']).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'测试集混淆矩阵\nAUC={test_auc:.4f}')

# ROC 曲线
fpr_v, tpr_v, _ = roc_curve(y_val, val_prob)
fpr_t, tpr_t, _ = roc_curve(y_test, test_prob)
axes[1].plot(fpr_v, tpr_v, 'b-', linewidth=2, label=f'验证集 AUC={val_auc:.3f}')
axes[1].plot(fpr_t, tpr_t, 'r-', linewidth=2, label=f'测试集 AUC={test_auc:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_title('ROC 曲线对比')
axes[1].set_xlabel('FPR')
axes[1].set_ylabel('TPR')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 指标对比表
metrics_df = pd.DataFrame({
    '指标': ['ROC-AUC', 'Average Precision', '准确率', '正样本率'],
    '验证集': [f'{val_auc:.4f}', f'{val_ap:.4f}',
               f'{(val_pred==y_val).mean():.4f}', f'{y_val.mean():.4f}'],
    '测试集': [f'{test_auc:.4f}', f'{test_ap:.4f}',
               f'{(test_pred==y_test).mean():.4f}', f'{y_test.mean():.4f}'],
})
axes[2].axis('off')
table = axes[2].table(
    cellText=metrics_df.values,
    colLabels=metrics_df.columns,
    loc='center', cellLoc='center'
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
axes[2].set_title('验证集 vs 测试集指标对比\n（差距小=未过拟合验证集）', pad=20)

plt.suptitle('步骤5: 逻辑回归模型评估（测试集 Holdout）', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

gap = abs(val_auc - test_auc)
print(f"\n✅ AUC 差距: {gap:.4f}", "（< 0.05 说明未过拟合验证集）" if gap < 0.05 else "⚠️（差距较大，注意过拟合）")

## 步骤 6：用模型在新标的上回测效果

**回测设计**：
- 使用 `oos_codes`（步骤2保留的 OOS 股票），这些股票**从未参与任何训练**
- 仅使用测试集时间段的 OOS 数据（严格 OOS）
- 信号：ML 预测概率 > `PROB_THRESHOLD` 时买入，持仓 `PRED_HORIZON` 天后卖出
- 与买入持有基准对比，评估模型带来的超额收益

**费率设置**（A股真实费率）:
- 手续费：买入 0.03%，卖出 0.03%（使用 `DEFAULT_COMMISSION_RATE`）
- 印花税：卖出 0.1%（使用 `DEFAULT_TAX_RATE`）
- 滑点：双向各 0.05%（使用 `DEFAULT_SLIPPAGE`）

In [ ]:
def build_oos_features(stock_df: pd.DataFrame) -> pd.DataFrame:
    """对 OOS 股票构建特征（使用相同的 build_features_single）"""
    return build_features_single(stock_df)


def generate_ml_signals(feat_df: pd.DataFrame,
                        model,
                        bounds_X_train: pd.DataFrame,
                        feature_cols: list,
                        prob_threshold: float = 0.55) -> pd.DataFrame:
    """
    生成 ML 交易信号。
    
    严格防穿越：
    - winsorization 边界来自训练集（X_train_w 的统计量）
    - 不重新训练任何参数
    """
    X = feat_df[feature_cols].copy().fillna(0)
    
    # 应用训练集的 winsorization（使用 X_train_w 的 min/max 作为代理边界）
    for col in feature_cols:
        if col in bounds_X_train.columns:
            lo = bounds_X_train[col].min()
            hi = bounds_X_train[col].max()
            X[col] = X[col].clip(lo, hi)
    
    probs = model.predict_proba(X)[:, 1]
    feat_df = feat_df.copy()
    feat_df['ml_prob'] = probs
    
    # 时间出场：持仓 PRED_HORIZON 天后出场
    entries = probs > prob_threshold
    
    # 构建出场信号：entry 后 PRED_HORIZON 天发出 exit
    exits = np.zeros(len(entries), dtype=bool)
    in_trade = False
    entry_idx = -1
    for i in range(len(entries)):
        if entries[i] and not in_trade:
            in_trade = True
            entry_idx = i
        elif in_trade and (i - entry_idx) >= PRED_HORIZON:
            exits[i] = True
            in_trade = False
    
    feat_df['entry_signal'] = entries
    feat_df['exit_signal']  = exits
    return feat_df

In [ ]:
def run_simple_backtest(feat_df: pd.DataFrame, init_cash=100_000,
                        buy_fee=DEFAULT_COMMISSION_RATE,
                        sell_fee=DEFAULT_COMMISSION_RATE + DEFAULT_TAX_RATE,
                        slippage=DEFAULT_SLIPPAGE):
    """
    简化版回测（不依赖 VectorBT 的纯 pandas 实现）。
    信号: entry_signal=True 时买入全仓；exit_signal=True 时卖出。
    返回: portfolio 统计字典
    """
    df = feat_df.set_index('date').sort_index()
    cash   = init_cash
    shares = 0
    equity = [init_cash]
    dates  = [df.index[0]]
    trades = []
    entry_price = 0
    
    for i in range(1, len(df)):
        price = df['close'].iloc[i]
        signal_entry = df['entry_signal'].iloc[i]
        signal_exit  = df['exit_signal'].iloc[i]
        
        if signal_entry and shares == 0 and cash > 0:
            # 买入
            buy_price = price * (1 + slippage)
            shares = int(cash * (1 - buy_fee) // buy_price)
            cost = shares * buy_price * (1 + buy_fee)
            cash -= cost
            entry_price = buy_price
        
        elif signal_exit and shares > 0:
            # 卖出
            sell_price = price * (1 - slippage)
            proceeds = shares * sell_price * (1 - sell_fee)
            pnl = proceeds - shares * entry_price
            trades.append({'pnl': pnl, 'return': pnl / (shares * entry_price)})
            cash += proceeds
            shares = 0
            entry_price = 0
        
        total = cash + shares * price
        equity.append(total)
        dates.append(df.index[i])
    
    equity_s = pd.Series(equity, index=dates)
    total_ret = equity_s.iloc[-1] / init_cash - 1
    
    # 年化
    years = (equity_s.index[-1] - equity_s.index[0]).days / 365.25
    ann_ret = (1 + total_ret) ** (1 / max(years, 0.1)) - 1
    
    # 最大回撤
    rolling_max = equity_s.cummax()
    drawdown    = (equity_s - rolling_max) / rolling_max
    max_dd      = drawdown.min()
    
    # 夏普（日收益）
    daily_ret = equity_s.pct_change().dropna()
    sharpe = daily_ret.mean() / (daily_ret.std() + 1e-10) * np.sqrt(252)
    
    win_rate = np.mean([t['return'] > 0 for t in trades]) if trades else 0
    
    return {
        'equity_curve': equity_s,
        'total_return': total_ret,
        'annualized_return': ann_ret,
        'sharpe_ratio': sharpe,
        'max_drawdown': max_dd,
        'win_rate': win_rate,
        'trade_count': len(trades),
    }


def run_vbt_backtest(feat_df: pd.DataFrame, init_cash=100_000):
    """使用 VectorBT 运行回测（若已安装）"""
    df = feat_df.set_index('date').sort_index()
    close   = df['close']
    entries = df['entry_signal']
    exits   = df['exit_signal']
    
    portfolio = vbt.Portfolio.from_signals(
        close=close,
        entries=entries,
        exits=exits,
        init_cash=init_cash,
        fees=DEFAULT_COMMISSION_RATE + DEFAULT_TAX_RATE,
        slippage=DEFAULT_SLIPPAGE,
        freq='1d'
    )
    eq = portfolio.value()
    return {
        'equity_curve':      eq,
        'total_return':      portfolio.total_return(),
        'annualized_return': portfolio.annualized_return(),
        'sharpe_ratio':      portfolio.sharpe_ratio(),
        'max_drawdown':      portfolio.max_drawdown(),
        'win_rate':          portfolio.trades.win_rate() if portfolio.trades.count() > 0 else 0,
        'trade_count':       portfolio.trades.count(),
    }

In [ ]:
test_start = splitter.test_start_date
print(f"📅 OOS 回测时间段: {test_start.date()} ~ {df_test['date'].max().date()}")
print(f"🔄 对 {len(oos_codes)} 只 OOS 股票生成信号并回测...\n")

oos_results = []
oos_equity_curves = {}

for code in oos_codes:
    stock_df = df_oos[df_oos['code'] == code].copy()
    
    # 构建特征（全时段，含足够的历史数据来计算指标）
    feat_df = build_oos_features(stock_df)
    if feat_df is None or len(feat_df) < 60:
        continue
    
    # 只取测试期用于回测
    feat_test = feat_df[feat_df['date'] >= test_start].copy()
    if len(feat_test) < 20:
        continue
    
    # 生成信号
    feat_test = generate_ml_signals(feat_test, best_model, X_train_w, 
                                     FEATURE_COLS, PROB_THRESHOLD)
    
    # 回测（优先 VectorBT）
    try:
        if HAS_VBT:
            result = run_vbt_backtest(feat_test)
        else:
            result = run_simple_backtest(feat_test)
    except Exception as e:
        result = run_simple_backtest(feat_test)
    
    # 买入持有基准
    prices = feat_test.set_index('date')['close'].sort_index()
    bh_return = prices.iloc[-1] / prices.iloc[0] - 1 if len(prices) > 1 else 0
    
    oos_results.append({
        'code':              code,
        'total_return':      result['total_return'],
        'annualized_return': result['annualized_return'],
        'sharpe_ratio':      result['sharpe_ratio'],
        'max_drawdown':      result['max_drawdown'],
        'win_rate':          result['win_rate'],
        'trade_count':       result['trade_count'],
        'bh_return':         bh_return,
        'excess_return':     result['total_return'] - bh_return,
    })
    oos_equity_curves[code] = result['equity_curve']

oos_df = pd.DataFrame(oos_results).sort_values('sharpe_ratio', ascending=False)
print(f"✅ 完成 {len(oos_df)} 只 OOS 股票回测")

In [ ]:
# 汇总统计
print("=" * 65)
print("📊 OOS 回测汇总统计")
print("=" * 65)
print(f"  回测股票数:              {len(oos_df)}")
print(f"  平均总收益率:            {oos_df['total_return'].mean():+.2%}")
print(f"  平均年化收益率:          {oos_df['annualized_return'].mean():+.2%}")
print(f"  平均 Sharpe:             {oos_df['sharpe_ratio'].mean():.3f}")
print(f"  平均最大回撤:            {oos_df['max_drawdown'].mean():.2%}")
print(f"  平均胜率:                {oos_df['win_rate'].mean():.1%}")
print(f"  平均超额收益（vs B&H）:  {oos_df['excess_return'].mean():+.2%}")
print(f"  跑赢买入持有比例:        {(oos_df['excess_return'] > 0).mean():.1%}")
print("=" * 65)

print("\n📋 各股票详细结果（按 Sharpe 排序，前15名）:")
display_cols = ['code', 'total_return', 'annualized_return', 'sharpe_ratio', 
                'max_drawdown', 'win_rate', 'trade_count', 'bh_return', 'excess_return']
oos_display = oos_df[display_cols].head(15).copy()
oos_display['total_return']      = oos_display['total_return'].map('{:+.2%}'.format)
oos_display['annualized_return'] = oos_display['annualized_return'].map('{:+.2%}'.format)
oos_display['sharpe_ratio']      = oos_display['sharpe_ratio'].map('{:.3f}'.format)
oos_display['max_drawdown']      = oos_display['max_drawdown'].map('{:.2%}'.format)
oos_display['win_rate']          = oos_display['win_rate'].map('{:.1%}'.format)
oos_display['bh_return']         = oos_display['bh_return'].map('{:+.2%}'.format)
oos_display['excess_return']     = oos_display['excess_return'].map('{:+.2%}'.format)
print(oos_display.to_string(index=False))

In [ ]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

# 1. 超额收益分布
ax1 = fig.add_subplot(gs[0, 0])
excess = oos_df['excess_return']
ax1.hist(excess.values, bins=20, color='steelblue', alpha=0.8, edgecolor='white')
ax1.axvline(0, color='red', linestyle='--', linewidth=1.5, label='零超额')
ax1.axvline(excess.mean(), color='orange', linestyle='-', linewidth=2, 
            label=f'均值={excess.mean():+.2%}')
ax1.set_title('超额收益分布（ML - 买入持有）')
ax1.set_xlabel('超额收益率')
ax1.set_ylabel('股票数量')
ax1.legend()

# 2. Sharpe 分布
ax2 = fig.add_subplot(gs[0, 1])
sharpe_vals = oos_df['sharpe_ratio']
ax2.hist(sharpe_vals.values, bins=20, color='#66BB6A', alpha=0.8, edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Sharpe=0')
ax2.axvline(sharpe_vals.mean(), color='orange', linestyle='-', linewidth=2,
            label=f'均值={sharpe_vals.mean():.3f}')
ax2.set_title('Sharpe 比率分布（OOS股票）')
ax2.set_xlabel('Sharpe 比率')
ax2.set_ylabel('股票数量')
ax2.legend()

# 3. ML收益 vs 买入持有 散点图
ax3 = fig.add_subplot(gs[1, 0])
ml_ret = oos_df['total_return'].values
bh_ret = oos_df['bh_return'].values
beat_bh = ml_ret > bh_ret
ax3.scatter(bh_ret[~beat_bh], ml_ret[~beat_bh], alpha=0.6, color='#EF5350', 
            label=f'跑输 ({(~beat_bh).sum()}只)')
ax3.scatter(bh_ret[beat_bh],  ml_ret[beat_bh],  alpha=0.6, color='#66BB6A',
            label=f'跑赢 ({beat_bh.sum()}只)')
lim = max(abs(bh_ret).max(), abs(ml_ret).max()) * 1.1
ax3.plot([-lim, lim], [-lim, lim], 'k--', alpha=0.5, label='对角线（等效）')
ax3.set_title('ML策略 vs 买入持有')
ax3.set_xlabel('买入持有收益率')
ax3.set_ylabel('ML策略收益率')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 权益曲线（展示部分OOS股票）
ax4 = fig.add_subplot(gs[1, 1])
top_codes = oos_df.head(min(5, len(oos_df)))['code'].tolist()
for code in top_codes:
    if code in oos_equity_curves:
        eq = oos_equity_curves[code]
        norm_eq = eq / eq.iloc[0]
        ax4.plot(norm_eq.index, norm_eq.values, linewidth=1.5, label=code, alpha=0.8)
ax4.axhline(1.0, color='black', linestyle='--', alpha=0.5, label='初始净值')
ax4.set_title('Top5 OOS股票权益曲线（归一化）')
ax4.set_xlabel('日期')
ax4.set_ylabel('净值')
ax4.legend(fontsize=7)
ax4.grid(True, alpha=0.3)
ax4.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)

plt.suptitle('步骤6: OOS回测结果（模型在新标的上的泛化效果）', 
             fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# 概率阈值敏感性分析
print("🔍 概率阈值敏感性分析...")
thresholds = [0.50, 0.52, 0.55, 0.58, 0.60, 0.63, 0.65]
threshold_results = []

for thresh in thresholds:
    res_list = []
    for code in oos_codes[:min(30, len(oos_codes))]:  # 取前30只快速测试
        stock_df = df_oos[df_oos['code'] == code].copy()
        feat_df  = build_oos_features(stock_df)
        if feat_df is None: continue
        feat_test = feat_df[feat_df['date'] >= test_start].copy()
        if len(feat_test) < 20: continue
        feat_test = generate_ml_signals(feat_test, best_model, X_train_w, 
                                         FEATURE_COLS, thresh)
        try:
            r = run_simple_backtest(feat_test)
            res_list.append(r)
        except: pass
    
    if res_list:
        avg_sharpe    = np.mean([r['sharpe_ratio']      for r in res_list])
        avg_excess    = np.mean([r['total_return']       for r in res_list])
        avg_trades    = np.mean([r['trade_count']        for r in res_list])
        threshold_results.append({
            'threshold': thresh,
            'avg_sharpe': avg_sharpe,
            'avg_excess': avg_excess,
            'avg_trades': avg_trades
        })

thresh_df = pd.DataFrame(threshold_results)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(thresh_df['threshold'], thresh_df['avg_sharpe'], 'bo-', linewidth=2)
axes[0].set_title('概率阈值 vs 平均 Sharpe')
axes[0].set_xlabel('概率阈值')
axes[0].set_ylabel('平均 Sharpe')
axes[0].grid(True, alpha=0.3)
axes[0].axvline(PROB_THRESHOLD, color='red', linestyle='--', label=f'当前={PROB_THRESHOLD}')
axes[0].legend()

axes[1].plot(thresh_df['threshold'], thresh_df['avg_excess'], 'go-', linewidth=2)
axes[1].set_title('概率阈值 vs 平均收益率')
axes[1].set_xlabel('概率阈值')
axes[1].set_ylabel('平均总收益率')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

axes[2].plot(thresh_df['threshold'], thresh_df['avg_trades'], 'ro-', linewidth=2)
axes[2].set_title('概率阈值 vs 平均交易次数')
axes[2].set_xlabel('概率阈值')
axes[2].set_ylabel('平均交易次数')
axes[2].grid(True, alpha=0.3)

plt.suptitle('阈值敏感性分析（阈值越高=信号越精 但交易越少）', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(thresh_df.to_string(index=False))

## 总结

### 流程回顾

| 步骤 | 完成内容 | 关键输出 |
|------|---------|----------|
| 1. 数据读取 | 从本地 CSV 加载 A 股历史数据 | `df_raw` |
| 2. 标的筛选 | 主板+排ST+数据充足，OOS保留20% | `ml_codes`, `oos_codes` |
| 3. 特征工程 | 40+技术因子，面板堆叠 | `panel_df`, `FEATURE_COLS` |
| 4. 数据划分 | 时序60/20/20，严格防穿越 | `X_train/val/test` |
| 5. 模型训练 | L2逻辑回归 + 验证集调参 | `best_model`, AUC指标 |
| 6. OOS回测 | VectorBT信号回测，vs买入持有 | `oos_df`, Sharpe等 |

### 防穿越检查清单

- [x] 目标标签 `shift(-N)` 在 split 前计算
- [x] `StandardScaler` 仅 fit 训练集
- [x] MAD 缩尾边界仅从训练集获取
- [x] 超参数用验证集（非 KFold）选择
- [x] OOS 股票从步骤2起完全隔离

### 扩展方向

1. **更多特征**: 换手率、资金流向、北向资金、财务因子
2. **更复杂模型**: XGBoost、LightGBM、LSTM（需更严格的时序CV）
3. **仓位管理**: Kelly 公式、最大化多样化（对多标的同时持仓）
4. **组合层面回测**: 同时持有概率最高的 Top-K 股票